# TP 2 — Intro Python (rappel express)

Exercices en lien avec le cours : `if`/`for`, chaînes, conteneurs, fonctions propres,
classes, exceptions en boucle, modules.

**Consignes :** type hints + docstring sur chaque fonction, exceptions précises
(`ValueError`), jamais de `except:` nu. Testez chaque exo en
exécutant sa cellule de tests.

## Ex 1 — Chaînes : logs (§2.2)

1. `est_echec(log: str) -> bool` : `True` si la ligne commence par `"failed"`
   (insensible à la casse, espaces autour ignorés). Méthodes : `strip()`, `lower()`,
   `startswith()`.
2. `formater_alerte(ip: str, regle: str) -> str` : f-string `"IP suspecte : <ip> [<regle>]"`.

In [ ]:
def est_echec(log: str) -> bool:
    """Renvoie True si la ligne de log signale un échec."""
    #BEGIN
    return log.strip().lower().startswith("failed")
    #END


def formater_alerte(ip: str, regle: str) -> str:
    """Renvoie 'IP suspecte : <ip> [<regle>]'."""
    #BEGIN
    return f"IP suspecte : {ip} [{regle}]"
    #END

In [ ]:
# Tests ex 1
assert est_echec("Failed password for root") is True
assert est_echec("  failed guest  ") is True
assert est_echec("ok admin") is False
assert formater_alerte("1.2.3.4", "brute-force SSH") == "IP suspecte : 1.2.3.4 [brute-force SSH]"
print("ex 1 OK")

## Ex 2 — Conteneurs : IOC (§2.3 + §3)

1. `dedupliquer(iocs: list[str]) -> list[str]` : renvoyer les IOC uniques **triés**
   (piste : `set` + `sorted`).
2. `filtrer_echecs(lignes: list[str]) -> list[str]` : en **une compréhension**, garder les
   lignes contenant `"failed"`.
3. `compter(lignes: list[str]) -> dict[str, int]` : compter les occurrences de chaque ligne
   (piste : `.get(cle, 0)`).

In [ ]:
def dedupliquer(iocs: list[str]) -> list[str]:
    """Renvoie les IOC uniques, triés."""
    #BEGIN
    return sorted(set(iocs))
    #END


def filtrer_echecs(lignes: list[str]) -> list[str]:
    """Garde les lignes contenant 'failed' (compréhension)."""
    #BEGIN
    return [l for l in lignes if "failed" in l]
    #END


def compter(lignes: list[str]) -> dict[str, int]:
    """Compte les occurrences de chaque ligne."""
    #BEGIN
    resultat: dict[str, int] = {}
    for ligne in lignes:
        resultat[ligne] = resultat.get(ligne, 0) + 1
    return resultat
    #END

In [ ]:
# Tests ex 2
assert dedupliquer(["5.6.7.8", "1.2.3.4", "1.2.3.4"]) == ["1.2.3.4", "5.6.7.8"]
assert dedupliquer([]) == []
assert filtrer_echecs(["ok admin", "failed root", "failed guest"]) == ["failed root", "failed guest"]
assert compter(["a", "b", "a"]) == {"a": 2, "b": 1}
print("ex 2 OK")

## Ex 3 — Classe `Alerte` (§6)

Compléter la classe : `__init__(ip, regle)` initialise `vues = 1`, `revoir()` incrémente,
`__str__` renvoie `"<ip> [<regle>] x<vues>"`.

In [ ]:
class Alerte:
    """Une alerte sur une IP suspecte."""

    def __init__(self, ip: str, regle: str):
        #BEGIN
        self.ip = ip
        self.regle = regle
        self.vues = 1
        #END

    def revoir(self) -> None:
        """Incrémente le compteur de vues."""
        #BEGIN
        self.vues += 1
        #END

    def __str__(self) -> str:
        #BEGIN
        return f"{self.ip} [{self.regle}] x{self.vues}"
        #END

In [ ]:
# Tests ex 3
a = Alerte("1.2.3.4", "brute-force SSH")
assert str(a) == "1.2.3.4 [brute-force SSH] x1"
a.revoir()
a.revoir()
assert str(a) == "1.2.3.4 [brute-force SSH] x3"
print("ex 3 OK")

## Ex 4 — Exceptions en boucle (§7)

`parser_ports(lignes: list[str]) -> tuple[list[int], int]` : convertir chaque ligne avec
`int()`. Le `try` est **dans** la boucle : une ligne pourrie est ignorée (`continue`)
sans arrêter l'analyse. Renvoyer `(valides, nb_erreurs)`. Attraper `ValueError` uniquement.

In [ ]:
def parser_ports(lignes: list[str]) -> tuple[list[int], int]:
    """Convertit les lignes en ports ; ignore les lignes invalides."""
    #BEGIN
    valides: list[int] = []
    erreurs: int = 0
    for ligne in lignes:
        try:
            valides.append(int(ligne))
        except ValueError:
            erreurs += 1
            continue
    return valides, erreurs
    #END

In [ ]:
# Tests ex 4
assert parser_ports(["22", "oups", "443"]) == ([22, 443], 1)
assert parser_ports([]) == ([], 0)
assert parser_ports(["x", "y"]) == ([], 2)
print("ex 4 OK")

## Ex 5 — Modules : `hashlib` (§8)

`sha256_texte(t: str) -> str` avec `hashlib` (cf. cours §8.1) : renvoyer
l'empreinte SHA256 hexadécimale de `t`.

In [ ]:
import hashlib
hashlib.sha256?

In [ ]:
def sha256_texte(t: str) -> str:
    """Renvoie l'empreinte SHA256 hexadécimale de t."""
    #BEGIN
    import hashlib
    return hashlib.sha256(t.encode()).hexdigest()
    #END

In [ ]:
# Tests ex 5
import hashlib

assert sha256_texte("abc") == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'
assert len(sha256_texte("x")) == 64
print("ex 5 OK")

## Bonus — IPs et mini-analyse de logs (tout combiner)

1. `est_ipv4_valide(s: str) -> bool` : `True` si `s` est une IPv4 valide —
   exactement 4 groupes séparés par des points, chaque groupe uniquement
   des chiffres (`isdigit()`) valant entre 0 et 255.
   Exemples : `"1.2.3.4"` → `True`, `"255.255.255.255"` → `True`,
   `"256.1.1.1"` → `False`, `"1.2.3"` → `False`, `"abc.def.ghi.jkl"` → `False`.
2. `extraire_ip(ligne: str) -> str | None` : renvoyer le dernier mot de la ligne
   si c'est une IPv4 valide (réutilisez `est_ipv4_valide`), sinon `None`.
   Ligne vide → `None`. Type hints + docstring obligatoires (cf. cours §5).
3. `analyser(lignes: list[str]) -> dict[str, int]` : pour chaque ligne, garder les échecs
   (`est_echec`), extraire l'IP (`extraire_ip`), compter les occurrences par IP.
   Réutilisez vos fonctions des ex 1 et du bonus (copiez-les dans la cellule si besoin).

In [ ]:
def est_ipv4_valide(s: str) -> bool:
    """Renvoie True si s est une adresse IPv4 valide (4 groupes 0-255)."""
    #BEGIN
    parties: list[str] = s.strip().split(".")
    if len(parties) != 4:
        return False
    for partie in parties:
        if not partie.isdigit():
            return False
        if not 0 <= int(partie) <= 255:
            return False
    return True
    #END


def extraire_ip(ligne: str) -> str | None:
    """Renvoie l'IP en fin de ligne de log (séparée par un espace), ou None si absente/invalide."""
    #BEGIN
    mots: list[str] = ligne.strip().split()
    if not mots:
        return None
    candidat: str = mots[-1]
    if est_ipv4_valide(candidat):
        return candidat
    return None
    #END


def analyser(lignes: list[str]) -> dict[str, int]:
    """Compte les échecs par IP."""
    #BEGIN
    resultat: dict[str, int] = {}
    for ligne in lignes:
        if not est_echec(ligne):
            continue
        ip = extraire_ip(ligne)
        if ip is None:
            continue
        resultat[ip] = resultat.get(ip, 0) + 1
    return resultat
    #END

In [ ]:
# Tests bonus
assert est_ipv4_valide("1.2.3.4") is True
assert est_ipv4_valide("255.255.255.255") is True
assert est_ipv4_valide("0.0.0.0") is True
assert est_ipv4_valide("256.1.1.1") is False
assert est_ipv4_valide("1.2.3") is False
assert est_ipv4_valide("abc.def.ghi.jkl") is False
assert est_ipv4_valide("") is False

assert extraire_ip("Failed password for root from 192.168.1.10") == "192.168.1.10"
assert extraire_ip("") is None
assert extraire_ip("pas d'ip ici") is None
assert extraire_ip("Failed from 999.1.1.1") is None

logs = [
    "Failed password for root from 1.2.3.4",
    "ok admin from 9.9.9.9",
    "FAILED password for guest from 1.2.3.4",
    "Failed password for admin from 5.6.7.8",
    "Failed sans ip",
]
assert analyser(logs) == {"1.2.3.4": 2, "5.6.7.8": 1}
print("bonus OK")

## Bonus 2 — Intégrité des logs : SHA256 tronqué (§8)

`detecter_modifications(logs: list[str], empreintes: list[str]) -> tuple[int, list[int]]` :
retourner un tuple `(nb, indices)` où `nb` est le nombre de lignes de log modifiées
et `indices` la liste des indices (0-based) de ces lignes, c'est-à-dire celles dont
le SHA256 tronqué ne correspond pas à l'empreinte attendue.

- `logs` et `empreintes` doivent avoir la même longueur, sinon lever `ValueError`.
- Renvoyer `(0, [])` si tout est intègre (y compris si les deux listes sont vides).
- Type hints + docstring obligatoires

In [ ]:
def detecter_modifications(logs: list[str], empreintes: list[str]) -> tuple[int, list[int]]:
    """Renvoie (nb_modifiées, indices) des lignes dont le SHA256 tronqué (10 car.) ne correspond pas."""
    #BEGIN
    import hashlib
    if len(logs) != len(empreintes):
        raise ValueError(f"longueurs différentes : {len(logs)} logs vs {len(empreintes)} empreintes")
    indices: list[int] = []
    for i, (ligne, empreinte) in enumerate(zip(logs, empreintes)):
        if hashlib.sha256(ligne.encode()).hexdigest()[:10] != empreinte:
            indices.append(i)
    return len(indices), indices
    #END

In [ ]:
# Tests bonus 2
logs_ok = ["Failed password for root from 1.2.3.4", "Accepted password for admin"]
empreintes_ok = ["9d78684a44", "425f70fa4d"]
assert detecter_modifications(logs_ok, empreintes_ok) == (0, [])
assert detecter_modifications([], []) == (0, [])

# Une ligne modifiée (indice 1)
logs_mod = ["Failed password for root from 1.2.3.4", "Accepted password for GUEST"]
assert detecter_modifications(logs_mod, empreintes_ok) == (1, [1])

# Tout modifié
assert detecter_modifications(["aaa", "bbb"], empreintes_ok) == (2, [0, 1])

# Longueurs différentes -> ValueError
try:
    detecter_modifications(["a"], [])
    raise AssertionError("ValueError attendue")
except ValueError:
    pass
print("bonus 2 OK")